# Health Study – Part 1: Descriptive Analysis and Statistics

## Introduction
In this notebook I analyse data from a health study with 800 participants.
The dataset contains age, sex, height, weight, systolic blood pressure,
cholesterol, smoking habits and whether the person has a certain disease.

The goals of Part 1 are to:
1. Describe the data with basic statistics and graphs.
2. Simulate the disease rate with numpy and compare it with the real data.
3. Calculate a confidence interval for the mean systolic blood pressure.
4. Test if smokers have higher blood pressure than non-smokers.

## 1. Setup
I import the libraries, set a random seed so that all simulations give
the same result every time, and load the data with a relative path.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ตั้ง seed = 42 ตามที่โจทย์กำหนด เพื่อให้สุ่มได้ผลเหมือนเดิมทุกครั้ง
np.random.seed(42)

df = pd.read_csv("data/health_study_dataset.csv")

df.head()

,id,age,sex,height,weight,systolic_bp,cholesterol,smoker,disease
0,1,57,F,168.9,65.8,141.8,4.58,No,0
1,2,47,M,180.4,95.9,144.8,5.18,Yes,0
2,3,59,F,169.9,82.2,151.7,6.16,No,0
3,4,72,M,157.7,93.1,151.0,6.63,No,0
4,5,46,M,192.6,104.1,144.1,5.21,No,0


## 2. Data quality check
Before the analysis I check the size of the data, the data types,
missing values and duplicates. Bad data gives bad results,
so this step is important.

In [2]:
# ขนาดข้อมูล (แถว, คอลัมน์)
print("Shape:", df.shape)

# ดูชนิดข้อมูลของแต่ละคอลัมน์
df.info()

# นับค่าที่หายไปในแต่ละคอลัมน์
print("\nMissing Values:", df.isnull().sum())

# นับแถวที่ซ้ำกัน
print("\nDuplicate Rows:", df.duplicated().sum())

Shape: (800, 9)
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           800 non-null    int64  
 1   age          800 non-null    int64  
 2   sex          800 non-null    str    
 3   height       800 non-null    float64
 4   weight       800 non-null    float64
 5   systolic_bp  800 non-null    float64
 6   cholesterol  800 non-null    float64
 7   smoker       800 non-null    str    
 8   disease      800 non-null    int64  
dtypes: float64(4), int64(3), str(2)
memory usage: 56.4 KB

Missing Values: id             0
age            0
sex            0
height         0
weight         0
systolic_bp    0
cholesterol    0
smoker         0
disease        0
dtype: int64

Duplicate Rows: 0


In [3]:
# ดูช่วงของค่าตัวเลขทั้งหมด
df.describe().round(2)

,id,age,height,weight,systolic_bp,cholesterol,disease
count,800.00,800.00,800.00,800.00,800.00,800.00,800.00
mean,400.50,49.43,171.85,73.41,149.18,4.93,0.06
std,231.08,14.50,9.80,13.69,12.79,0.85,0.24
min,1.00,18.00,144.40,33.70,106.80,2.50,0.00
25%,200.75,39.00,164.78,64.80,140.90,4.33,0.00
50%,400.50,50.00,171.35,73.20,149.40,4.97,0.00
75%,600.25,59.00,178.92,82.60,157.60,5.48,0.00
max,800.00,90.00,200.40,114.40,185.90,7.88,1.00


In [4]:
# เรียงตามน้ำหนักจากน้อยไปมาก ดู 5 คนที่เบาที่สุด
lightest = df.sort_values("weight").head(5)

# คำนวณ BMI = น้ำหนัก(kg) / ส่วนสูง(m)^2 เพื่อดูว่าน้ำหนักสมเหตุสมผลกับส่วนสูงไหม
lightest_bmi = lightest["weight"] / (lightest["height"] / 100) ** 2

print(lightest[["age", "sex", "height", "weight"]])
print("\nBMI:", lightest_bmi.round(1).values)

     age sex  height  weight
709   75   F   149.6    33.7
535   50   F   163.8    33.9
508   50   F   155.9    35.5
23    28   F   170.2    38.1
388   64   F   170.5    41.6

BMI: [15.1 12.6 14.6 13.2 14.3]


## 3. Descriptive statistics
I calculate the mean, median, min and max for the five numeric columns.
The mean shows the average value, and the median shows the middle value.
If they are close, the data is not very skewed.

In [5]:
# คอลัมน์ที่โจทย์ให้คำนวณ
num_cols = ["age", "weight", "height", "systolic_bp", "cholesterol"]

# คำนวณ 4 ค่าพร้อมกันด้วย agg แล้วกลับตาราง (.T) ให้แต่ละคอลัมน์เป็นแถว อ่านง่ายกว่า
summary = df[num_cols].agg(["mean", "median", "min", "max"]).T
summary.round(2)

,mean,median,min,max
age,49.43,50.00,18.0,90.00
weight,73.41,73.20,33.7,114.40
height,171.85,171.35,144.4,200.40
systolic_bp,149.18,149.40,106.8,185.90
cholesterol,4.93,4.97,2.5,7.88


**Comment:** The mean and median are almost the same for all columns.
The average participant is about 49 years old. The mean systolic blood
pressure is 149 mmHg, which is above the common limit for high blood
pressure (140 mmHg). This means that many participants in this study
have high blood pressure.